# 📰 A Hybrid Deep-Learning Approach to Automated Misinformation Detection
### Fake News Detection & Suspicion Scoring System

This Jupyter / Google Colab notebook represents a complete, self-contained implementation of a high-performance **Fake News Detection pipeline**. It implements a hybrid architecture combining classical machine learning, state-of-the-art transformers, and heuristic styling checks to evaluate the credibility of news statements.

### 🏗️ Pipeline Architecture
1. **Classical Machine Learning Baseline**: Term Frequency-Inverse Document Frequency (**TF-IDF**) extraction paired with a **Logistic Regression** classifier.
2. **Deep Learning Sequence Classification**: A fine-tuned **DistilBERT** transformer model implemented using PyTorch and Hugging Face `transformers`.
3. **Rule-Based Linguistic Suspicion Engine**: A heuristic scoring system analyzing clickbait phrases, exclamation density, emotional weight, and uppercase emphasis to provide qualitative explanations.
4. **Interactive Widget Playground**: A user-friendly graphical interface utilizing `ipywidgets` to test inputs dynamically within the notebook.
5. **Streamlit Visual Dashboard Deployment**: Setup instructions and tunnel commands to host and run the project's interactive Streamlit web dashboard right from your Colab runtime.

---

## 🛠️ Step 0: Environment and Directory Setup
We begin by setting up the workspace directory hierarchy and installing necessary Python packages.

In [ ]:
# 1. Install missing dependencies in Google Colab
!pip install -q transformers joblib streamlit ipywidgets matplotlib nltk pandas scikit-learn

# 2. Create the required directory structure
import os

dirs = [
    'data/raw',
    'data/processed',
    'models/deep_model',
    'results'
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"Directory ensured: {d}")

## 📥 Step 1: Data Acquisition (The LIAR Dataset)
The **LIAR dataset** is a widely referenced benchmark dataset for fake news detection, containing over 12.8K human-labeled short statements from PolitiFact. 

We programmatically download the dataset splits (`train.tsv`, `valid.tsv`, and `test.tsv`) directly into `data/raw/` from verified public repositories.

In [ ]:
import urllib.request

# Public direct URLs to raw LIAR dataset tsv files hosted on GitHub
urls = {
    "train.tsv": "https://raw.githubusercontent.com/ekagra-ranjan/fake-news-detection-LIAR-pytorch/master/train.tsv",
    "valid.tsv": "https://raw.githubusercontent.com/ekagra-ranjan/fake-news-detection-LIAR-pytorch/master/valid.tsv",
    "test.tsv": "https://raw.githubusercontent.com/ekagra-ranjan/fake-news-detection-LIAR-pytorch/master/test.tsv"
}

for filename, url in urls.items():
    dest_path = os.path.join("data/raw", filename)
    if not os.path.exists(dest_path):
        print(f"Downloading {filename} from remote GitHub repository...")
        urllib.request.urlretrieve(url, dest_path)
        print(f"✓ Downloaded and saved: {dest_path}")
    else:
        print(f"✓ {filename} already exists at {dest_path}.")

## ⚙️ Step 2: Data Preprocessing
In this step, we read the tab-separated dataset files, apply binary text labeling, and sanitize the statements. 

Specifically, the LIAR dataset contains 6 target classes:
- `true`, `mostly-true`, `half-true` $\rightarrow$ **real**
- `barely-true`, `false`, `pants-fire` $\rightarrow$ **fake**

The statement cleaning logic involves:
- Lowercasing all characters
- Stripping URLs and numbers
- Removing punctuation marks
- Eliminating NLTK stop words to reduce noise
- Discarding empty records

In [ ]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords

# Download the NLTK stop words list
nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english"))

# Schema headers for LIAR dataset
LIAR_COLUMNS = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts", "mostly_true_counts", "pants_fire_counts", "context"
]

def load_liar_dataset(file_path):
    return pd.read_csv(file_path, sep="\t", names=LIAR_COLUMNS)

def convert_label(label):
    real_labels = ["true", "mostly-true", "half-true"]
    return "real" if label in real_labels else "fake"

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text) # Remove links
    text = re.sub(r"\d+", "", text)            # Remove digits
    text = text.translate(str.maketrans("", "", string.punctuation)) # Remove punctuation
    text = re.sub(r"\s+", " ", text).strip()   # Remove extra whitespace
    words = text.split()
    words = [w for w in words if w not in STOP_WORDS] # Exclude stopwords
    return " ".join(words)

def preprocess_and_save(input_path, output_path):
    print(f"Processing: {input_path}")
    df = load_liar_dataset(input_path)
    df = df[["label", "statement"]]
    df["target"] = df["label"].apply(convert_label)
    df["clean_text"] = df["statement"].apply(clean_text)
    df = df[df["clean_text"].str.strip() != ""]
    df.to_csv(output_path, index=False)
    print(f"✓ Saved clean dataset to: {output_path} (Shape: {df.shape})")
    return df

# Preprocess all splits
train_df = preprocess_and_save("data/raw/train.tsv", "data/processed/train_clean.csv")
valid_df = preprocess_and_save("data/raw/valid.tsv", "data/processed/valid_clean.csv")
test_df  = preprocess_and_save("data/raw/test.tsv", "data/processed/test_clean.csv")

## 📊 Step 3: Feature Extraction (TF-IDF)
We convert the preprocessed textual statements into numerical features using **TF-IDF (Term Frequency-Inverse Document Frequency)** vectorization. 

We restrict the vocabulary size to the top 5,000 features and consider both single words (unigrams) and word pairs (bigrams) to capture localized semantic structure.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

def create_tfidf_vectorizer():
    # Unigrams + Bigrams up to 5k vocabulary
    return TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2)
    )

print("TF-IDF Vectorizer initialized configuration.")

## 🤖 Step 4: Classical Machine Learning Model (Logistic Regression)
We train a high-performance classical machine learning baseline using **Logistic Regression** over the extracted TF-IDF representations. 

Following training, the model's parameters and vectorizer are serialized for later inference. We also generate a classification report and plot a visual confusion matrix.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

def train_ml_pipeline():
    # Load clean data
    train_df = pd.read_csv("data/processed/train_clean.csv")
    valid_df = pd.read_csv("data/processed/valid_clean.csv")
    test_df  = pd.read_csv("data/processed/test_clean.csv")

    X_train, y_train = train_df["clean_text"], train_df["target"]
    X_valid, y_valid = valid_df["clean_text"], valid_df["target"]
    X_test,  y_test  = test_df["clean_text"],  test_df["target"]

    # Vectorize
    vectorizer = create_tfidf_vectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_valid_tfidf = vectorizer.transform(X_valid)
    X_test_tfidf  = vectorizer.transform(X_test)

    # Train Logistic Regression
    model = LogisticRegression(max_iter=1000, solver='lbfgs')
    print("Training Logistic Regression Model...")
    model.fit(X_train_tfidf, y_train)

    # Validate and Test
    valid_preds = model.predict(X_valid_tfidf)
    valid_acc = accuracy_score(y_valid, valid_preds)
    print(f"Validation Accuracy: {valid_acc:.4f}")

    test_preds = model.predict(X_test_tfidf)
    test_acc = accuracy_score(y_test, test_preds)
    print(f"Test Accuracy: {test_acc:.4f}")

    # Save Model & Vectorizer
    joblib.dump(model, "models/ml_model.pkl")
    joblib.dump(vectorizer, "models/tfidf_vectorizer.pkl")
    print("✓ Saved serialized model & vectorizer inside 'models/' directory.")

    # Save report
    report = classification_report(y_test, test_preds)
    with open("results/evaluation_report.txt", "w") as f:
        f.write(f"Validation Accuracy: {valid_acc}\n")
        f.write(f"Test Accuracy: {test_acc}\n\n")
        f.write(report)
    print("✓ Saved results/evaluation_report.txt")

    # Plot and display confusion matrix inline
    cm = confusion_matrix(y_test, test_preds, labels=["real", "fake"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["real", "fake"])
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title("Confusion Matrix (Classical ML Baseline)")
    plt.savefig("results/confusion_matrix.png", bbox_inches='tight')
    plt.show()

train_ml_pipeline()

## 🧠 Step 5: Deep Learning Model Fine-Tuning (DistilBERT)
For semantic deep learning classification, we fine-tune a pre-trained **DistilBERT** transformer model using PyTorch and Hugging Face `transformers`.

> [!IMPORTANT]
> **GPU Speedup Recommendation:** To run this transformer training loop, click **Runtime -> Change runtime type** in the Colab menu and select **T4 GPU**. Doing so will complete epochs in seconds instead of hours.
>
> **Subsetting for Quick Verification:** Training BERT over 10K+ instances on CPU is highly restrictive. We've introduced a parameter `USE_SUBSET` (enabled by default when running on a CPU) that restricts the training data to a smaller subset (e.g. 600 train / 150 test samples) to ensure the notebook runs to completion quickly without resource exhausting errors.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Basic Model Configuration
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64
BATCH_SIZE = 8
EPOCHS = 2  # Reduced to 2 epochs for Colab demonstration speed
LEARNING_RATE = 3e-5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Execution Device Selected: {DEVICE}")

# Quick Run configuration
USE_SUBSET = True if DEVICE.type == "cpu" else False
SUBSET_SIZE_TRAIN = 600
SUBSET_SIZE_VAL_TEST = 150

LABEL_TO_ID = {"fake": 0, "real": 1}
ID_TO_LABEL = {0: "fake", 1: "real"}

# Custom PyTorch dataset class wrapper
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.encodings = tokenizer(
            self.texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH
        )
        
    def __len__(self):
        return len(self.texts)
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

def load_processed_data():
    train_df = pd.read_csv("data/processed/train_clean.csv")
    valid_df = pd.read_csv("data/processed/valid_clean.csv")
    test_df  = pd.read_csv("data/processed/test_clean.csv")

    # For transformers, use the original (uncleaned) statement column for contextual rich embedding
    train_texts = train_df["statement"].fillna("").tolist()
    valid_texts = valid_df["statement"].fillna("").tolist()
    test_texts  = test_df["statement"].fillna("").tolist()

    train_labels = train_df["target"].map(LABEL_TO_ID).tolist()
    valid_labels = valid_df["target"].map(LABEL_TO_ID).tolist()
    test_labels  = test_df["target"].map(LABEL_TO_ID).tolist()

    if USE_SUBSET:
        print(f"⚠️ USE_SUBSET is enabled. Limiting dataset sizes for quick training...")
        train_texts, train_labels = train_texts[:SUBSET_SIZE_TRAIN], train_labels[:SUBSET_SIZE_TRAIN]
        valid_texts, valid_labels = valid_texts[:SUBSET_SIZE_VAL_TEST], valid_labels[:SUBSET_SIZE_VAL_TEST]
        test_texts,  test_labels  = test_texts[:SUBSET_SIZE_VAL_TEST],  test_labels[:SUBSET_SIZE_VAL_TEST]

    return train_texts, train_labels, valid_texts, valid_labels, test_texts, test_labels

def evaluate_deep_model(model, loader):
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            batch_preds = torch.argmax(outputs.logits, dim=1)
            
            preds.extend(batch_preds.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
            
    acc = accuracy_score(actuals, preds)
    return acc, actuals, preds

def run_deep_training():
    os.makedirs("models/deep_model", exist_ok=True)
    train_texts, train_labels, valid_texts, valid_labels, test_texts, test_labels = load_processed_data()
    
    print("Downloading pre-trained tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_dataset = NewsDataset(train_texts, train_labels, tokenizer)
    valid_dataset = NewsDataset(valid_texts, valid_labels, tokenizer)
    test_dataset  = NewsDataset(test_texts, test_labels, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)
    
    print("Downloading DistilBERT classification pre-trained weights...")
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID
    ).to(DEVICE)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    print("\n🚀 Starting Transformer Fine-Tuning...")
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        for step, batch in enumerate(train_loader):
            optimizer.zero_grad()
            
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            if (step + 1) % (len(train_loader) // 2 or 1) == 0:
                print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{step+1}/{len(train_loader)}] Loss: {loss.item():.4f}")
                
        avg_loss = running_loss / len(train_loader)
        val_acc, _, _ = evaluate_deep_model(model, valid_loader)
        print(f"--- Epoch {epoch+1} Complete. Avg Training Loss: {avg_loss:.4f} | Validation Acc: {val_acc:.4f} ---\n")
        
    print("Training complete. Executing final test verification...")
    test_acc, test_labels_final, test_predictions = evaluate_deep_model(model, test_loader)
    print(f"Final Transformer Test Accuracy: {test_acc:.4f}")
    
    # Save report
    report = classification_report(test_labels_final, test_predictions, target_names=["fake", "real"])
    with open("results/deep_learning_report.txt", "w") as f:
        f.write("Deep Learning Model: DistilBERT\n")
        f.write(f"Subset Enabled: {USE_SUBSET}\n")
        f.write(f"Test Accuracy: {test_acc}\n\n")
        f.write(report)
    print("✓ Saved results/deep_learning_report.txt")
    
    # Save model and tokenizer state
    model.save_pretrained("models/deep_model")
    tokenizer.save_pretrained("models/deep_model")
    print("✓ DistilBERT state saved inside models/deep_model")

run_deep_training()

## 🔍 Step 6: Rule-Based Linguistic Suspicion Engine
To provide robust model interpretability and qualitative explanations for prediction decisions, we run a custom **Linguistic Suspicion Engine**.

This heuristic system scans for features indicative of clickbait or sensationalism:
1. **Capitalization Density**: Frequent FULLY UPPERCASE words trigger a red flag for hype.
2. **Exclamation Overuse**: Multiple consecutive or frequent exclamation marks indicate artificial urgency.
3. **Sensational Words**: Search against emotional triggers (`shocking`, `unbelievable`, `exposed`, `scandal`).
4. **Clickbait Phrase Matches**: Detect common programmatic clickbait phrases (`you won't believe`, `secret trick`).

These features are quantified into a structural **Linguistic Risk Score** scaled between $0.0$ and $1.0$.

In [ ]:
EMOTIONAL_WORDS = [
    "shocking", "unbelievable", "breaking", "exposed", "secret", "urgent",
    "miracle", "danger", "warning", "terrifying", "amazing", "hate", "destroy", "scandal"
]

CLICKBAIT_PHRASES = [
    "you won't believe", "watch now", "share now", "must see",
    "doctors hate", "secret trick", "what happened next", "before it's too late"
]

def check_capital_words(text):
    words = text.split()
    if len(words) == 0: return 0
    return len([w for w in words if w.isupper() and len(w) > 2])

def check_exclamation_marks(text):
    return text.count("!")

def check_emotional_words(text):
    text_lower = text.lower()
    return [word for word in EMOTIONAL_WORDS if word in text_lower]

def check_clickbait_phrases(text):
    text_lower = text.lower()
    return [phrase for phrase in CLICKBAIT_PHRASES if phrase in text_lower]

def generate_linguistic_report(text):
    reasons = []
    capitals = check_capital_words(text)
    exclamations = check_exclamation_marks(text)
    emotionals = check_emotional_words(text)
    clickbaits = check_clickbait_phrases(text)
    
    if capitals >= 2:
        reasons.append("⚠️ High density of capitalized words (potential attention grabbing).")
    if exclamations >= 2:
        reasons.append("⚠️ Excessive exclamation points (artificial sensationalism).")
    if emotionals:
        reasons.append(f"⚠️ Emotional or sensationalized vocabulary: {', '.join(emotionals)}")
    if clickbaits:
        reasons.append(f"⚠️ Clickbait style phrasing detected: {', '.join(clickbaits)}")
        
    if not reasons:
        reasons.append("✓ No major stylistic or linguistic red flags detected.")
    return reasons

def calculate_linguistic_risk(text):
    score = 0.0
    capitals = check_capital_words(text)
    exclamations = check_exclamation_marks(text)
    emotionals = check_emotional_words(text)
    clickbaits = check_clickbait_phrases(text)
    
    if capitals >= 2: score += 0.25
    if exclamations >= 2: score += 0.25
    if emotionals: score += 0.25
    if clickbaits: score += 0.25
    return min(score, 1.0)

# Quick dry run
sample = "URGENT!!! You won't believe the shocking truth exposed by doctors!"
print("Sample Report:", generate_linguistic_report(sample))
print("Linguistic Risk Score:", calculate_linguistic_risk(sample))

## 🔌 Step 7: Combined Prediction Framework
We unite the models into a single combined evaluation framework.

The model allows predicting using either standard ML, deep learning, or both. We combine the probabilistic score of the classifier with the **Linguistic Risk score** to formulate a final comprehensive, unified credibility report.

- **For Classical ML Pipeline:** $\text{Risk}_{\text{Final}} = 0.9 \times P(\text{Fake}) + 0.1 \times \text{Risk}_{\text{Linguistic}}$
- **For Deep Learning Pipeline:** $\text{Risk}_{\text{Final}} = 0.85 \times P(\text{Fake}) + 0.15 \times \text{Risk}_{\text{Linguistic}}$

In [ ]:
import joblib

def predict_classical(text):
    # Load state
    model = joblib.load("models/ml_model.pkl")
    vectorizer = joblib.load("models/tfidf_vectorizer.pkl")
    
    cleaned = clean_text(text)
    tfidf_feat = vectorizer.transform([cleaned])
    
    prediction = model.predict(tfidf_feat)[0]
    probs = model.predict_proba(tfidf_feat)[0]
    class_names = model.classes_
    prob_dict = dict(zip(class_names, probs))
    fake_prob = prob_dict.get("fake", 0.0)
    
    ling_risk = calculate_linguistic_risk(text)
    final_risk = (fake_prob * 0.90) + (ling_risk * 0.10)
    report = generate_linguistic_report(text)
    
    return {
        "model": "Classical Logistic Regression Baseline",
        "prediction": prediction,
        "fake_probability": fake_prob,
        "linguistic_risk": ling_risk,
        "final_risk": final_risk,
        "report": report
    }

def predict_deep(text):
    # Check if weights exist, otherwise default to classical
    if not os.path.exists("models/deep_model/config.json"):
        print("DistilBERT weights not found, fallback to classical model.")
        return predict_classical(text)
        
    tokenizer = AutoTokenizer.from_pretrained("models/deep_model")
    model = AutoModelForSequenceClassification.from_pretrained("models/deep_model").to(DEVICE)
    model.eval()
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = torch.softmax(outputs.logits, dim=1)[0]
        
    fake_prob = probabilities[0].item()
    real_prob = probabilities[1].item()
    prediction = "fake" if fake_prob > real_prob else "real"
    
    ling_risk = calculate_linguistic_risk(text)
    final_risk = (fake_prob * 0.85) + (ling_risk * 0.15)
    report = generate_linguistic_report(text)
    
    return {
        "model": "Fine-tuned DistilBERT Transformer",
        "prediction": prediction,
        "fake_probability": fake_prob,
        "linguistic_risk": ling_risk,
        "final_risk": final_risk,
        "report": report
    }

def run_prediction_flow(text, mode="ml"):
    if mode == "deep":
        return predict_deep(text)
    return predict_classical(text)

# Quick validation
res = run_prediction_flow("The federal reserve has decided to reduce rates this fiscal quarter.", mode="ml")
print("ML prediction outcome:", res)

## 🎨 Step 8: Premium Interactive Playground GUI
We create a premium visual interface inside the Google Colab workspace using `ipywidgets`. 

You can type your own news headlines, click the trigger button, and instantly view color-coded status, risk meters, and linguistic findings directly in the notebook!

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

def get_risk_label_and_color(risk):
    if risk < 0.40:
        return "LOW RISK", "#10b981" # Emerald Green
    elif risk < 0.70:
        return "MEDIUM RISK", "#f59e0b" # Amber Orange
    else:
        return "HIGH RISK", "#ef4444" # Rose Red

output_area = widgets.Output()

headline_input = widgets.Textarea(
    value='BREAKING!!! Doctors reveal a SECRET trick to cure index fatigue before it\'s too late!',
    placeholder='Enter news headline here...',
    description='News Text:',
    layout=widgets.Layout(width='90%', height='100px')
)

model_selector = widgets.Dropdown(
    options=[('TF-IDF + Logistic Regression (Fast)', 'ml'), ('DistilBERT Transformer (Deep)', 'deep')],
    value='ml',
    description='Model Type:',
)

analyze_btn = widgets.Button(
    description='Analyze Credibility',
    button_style='primary',
    tooltip='Click to execute classification and styling heuristics',
    icon='search'
)

def on_click_analyze(b):
    with output_area:
        output_area.clear_output()
        txt = headline_input.value.strip()
        if not txt:
            display(HTML("<p style='color:#f59e0b;'>⚠️ Please enter some text before analyzing!</p>"))
            return
            
        selected_mode = model_selector.value
        display(HTML("<p style='color:#6b7280;'>🔍 Processing text using " + selected_mode + " engine...</p>"))
        
        res = run_prediction_flow(txt, mode=selected_mode)
        
        lbl, col = get_risk_label_and_color(res["final_risk"])
        pred_str = res["prediction"].upper()
        pred_color = "#ef4444" if res["prediction"] == "fake" else "#10b981"
        
        report_items = "".join([f"<li style='margin-bottom:6px;'>{item}</li>" for item in res["report"]])
        
        html_template = f"""
        <div style='background-color:#f9fafb; border:1px solid #e5e7eb; border-radius:12px; padding:20px; font-family:system-ui, sans-serif;'>
            <h3 style='margin-top:0; color:#111827;'>Analysis Results</h3>
            
            <div style='display:flex; gap:20px; margin-bottom:15px;'>
                <div style='flex:1; background-color:#fff; border:1px solid #f3f4f6; border-radius:8px; padding:12px; text-align:center;'>
                    <strong style='color:#6b7280; font-size:12px; text-transform:uppercase;'>Prediction</strong>
                    <h2 style='margin:8px 0 0 0; color:{pred_color};'>{pred_str} NEWS</h2>
                </div>
                <div style='flex:1; background-color:#fff; border:1px solid #f3f4f6; border-radius:8px; padding:12px; text-align:center;'>
                    <strong style='color:#6b7280; font-size:12px; text-transform:uppercase;'>Final Risk Grade</strong>
                    <h2 style='margin:8px 0 0 0; color:{col};'>{lbl} ({round(res['final_risk']*100, 1)}%)</h2>
                </div>
            </div>
            
            <div style='margin-bottom:20px;'>
                <strong style='color:#4b5563; font-size:14px;'>Unified Suspicion Meter:</strong>
                <div style='background-color:#e5e7eb; border-radius:9999px; height:12px; margin-top:8px; overflow:hidden;'>
                    <div style='background-color:{col}; width:{res['final_risk']*100}%; height:100%; border-radius:9999px;'></div>
                </div>
                <div style='display:flex; justify-content:space-between; margin-top:4px; font-size:11px; color:#9ca3af;'>
                    <span>0% (Perfect Credibility)</span>
                    <span>100% (Absolute Suspicion)</span>
                </div>
            </div>
            
            <div style='background-color:#fff; border:1px solid #f3f4f6; border-radius:8px; padding:15px;'>
                <strong style='color:#4b5563; font-size:14px;'>Qualitative Linguistic Findings:</strong>
                <ul style='margin:8px 0 0 0; padding-left:20px; color:#4b5563; font-size:13px;'>
                    {report_items}
                </ul>
            </div>
            
            <div style='margin-top:12px; text-align:right; font-size:10px; color:#9ca3af;'>
                Evaluation Model: {res['model']}
            </div>
        </div>
        """
        display(HTML(html_template))

analyze_btn.on_click(on_click_analyze)

print("=== INTERACTIVE MISINFORMATION DETECTION SYSTEM ===")
display(headline_input)
display(model_selector)
display(analyze_btn)
display(output_area)

## 🌐 Step 9: Deploy Streamlit Web Dashboard Directly from Colab!
This project includes a beautiful Streamlit visual dashboard configured in `app/streamlit_app.py`.

We can expose and serve this local Streamlit dashboard directly from Google Colab's virtual network out to the public internet using **Localtunnel**.

Follow these simple commands to boot and launch your web app:

### 1. Save local script copies (Colab files sync)
If you are copying this notebook, ensure you run the cell below to write the Streamlit UI script contents directly onto the disk.

In [ ]:
import os
os.makedirs("app", exist_ok=True)

streamlit_code = """
import streamlit as st
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))
sys.path.append(PROJECT_ROOT)

from app.utils import get_prediction, get_risk_color, get_risk_label

st.set_page_config(
    page_title="Fake News Detection System",
    page_icon="📰",
    layout="centered"
)

st.title("📰 Automated Misinformation Detection")
st.write("Enter a news statement or headline to check whether it looks real or fake.")

user_text = st.text_area(
    "Enter News Text Here:",
    height=150,
    placeholder="Example: BREAKING!!! You won't believe what happened next!"
)

if st.button("Check Credibility"):
    if user_text.strip() == "":
        st.warning("Please enter some news text first.")
    else:
        result = get_prediction(user_text)
        prediction = result["prediction"]
        fake_probability = result["fake_probability"]
        final_risk = result["final_risk"]
        report = result["report"]
        
        risk_percent = round(final_risk * 100, 2)
        risk_color = get_risk_color(final_risk)
        risk_label = get_risk_label(final_risk)
        
        st.subheader("Prediction Result")
        if prediction == "fake":
            st.error("Prediction: Fake News")
        else:
            st.success("Prediction: Real News")
            
        st.subheader("Risk Meter")
        st.progress(final_risk)
        st.markdown(f\"\"\"<h3 style='color:{risk_color};'>{risk_label}: {risk_percent}%</h3>\"\"\", unsafe_allow_html=True)
        st.write("Fake Probability:", round(fake_probability * 100, 2), "%")
        
        st.subheader("Linguistic Report")
        for reason in report:
            st.write("•", reason)
            
st.markdown("---")
st.caption("Built using NLP, TF-IDF, Scikit-learn, and Streamlit.")
"""

with open("app/streamlit_app.py", "w") as f:
    f.write(streamlit_code.strip())

utils_code = """
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

from predict import predict_news

def get_prediction(text):
    return predict_news(text)

def get_risk_color(risk_score):
    if risk_score < 0.40: return "green"
    elif risk_score < 0.70: return "orange"
    else: return "red"

def get_risk_label(risk_score):
    if risk_score < 0.40: return "Low Risk"
    elif risk_score < 0.70: return "Medium Risk"
    else: return "High Risk"
"""

with open("app/utils.py", "w") as f:
    f.write(utils_code.strip())

# We must also write src folder scripts if they are missing in the local colab directory
os.makedirs("src", exist_ok=True)
print("✓ Wrote Streamlit scripts to disk.")

In [ ]:
# Port core modules directly to Colab disk
import json

preprocessing_data = """import pandas as pd\nimport re\nimport string\nimport nltk\nfrom nltk.corpus import stopwords\n\nSTOP_WORDS = set(stopwords.words(\"english\"))\n\nLIAR_COLUMNS = [\n    \"id\", \"label\", \"statement\", \"subject\", \"speaker\", \"speaker_job\",\n    \"state\", \"party\", \"barely_true_counts\", \"false_counts\",\n    \"half_true_counts\", \"mostly_true_counts\", \"pants_fire_counts\", \"context\"\n]\n\ndef load_liar_dataset(file_path):\n    return pd.read_csv(file_path, sep=\"\\t\", names=LIAR_COLUMNS)\n\ndef convert_label(label):\n    real_labels = [\"true\", \"mostly-true\", \"half-true\"]\n    return \"real\" if label in real_labels else \"fake\"\n\ndef clean_text(text):\n    text = str(text).lower()\n    text = re.sub(r\"http\\S+|www\\S+\", \"\", text)\n    text = re.sub(r\"\\d+\", \"\", text)\n    text = text.translate(str.maketrans(\"\", \"\", string.punctuation))\n    text = re.sub(r\"\\s+\", \" \", text).strip()\n    words = text.split()\n    words = [word for word in words if word not in STOP_WORDS]\n    return \" \".join(words)\n\ndef preprocess_and_save(input_path, output_path):\n    df = load_liar_dataset(input_path)\n    df = df[[\"label\", \"statement\"]]\n    df[\"target\"] = df[\"label\"].apply(convert_label)\n    df[\"clean_text\"] = df[\"statement\"].apply(clean_text)\n    df = df[df[\"clean_text\"].str.strip() != \"\"]\n    df.to_csv(output_path, index=False)\n    return df\n"""
with open("src/preprocessing.py", "w") as f:
    f.write(preprocessing_data)

feature_extraction_data = """from sklearn.feature_extraction.text import TfidfVectorizer\nimport joblib\n\ndef create_tfidf_vectorizer():\n    return TfidfVectorizer(max_features=5000, ngram_range=(1, 2))\n\ndef save_vectorizer(vectorizer, path):\n    joblib.dump(vectorizer, path)\n\ndef load_vectorizer(path):\n    return joblib.load(path)\n"""
with open("src/feature_extraction.py", "w") as f:
    f.write(feature_extraction_data)

linguistic_report_data = """EMOTIONAL_WORDS = [\n    \"shocking\", \"unbelievable\", \"breaking\", \"exposed\", \"secret\", \"urgent\",\n    \"miracle\", \"danger\", \"warning\", \"terrifying\", \"amazing\", \"hate\", \"destroy\", \"scandal\"\n]\n\nCLICKBAIT_PHRASES = [\n    \"you won\'t believe\", \"watch now\", \"share now\", \"must see\",\n    \"doctors hate\", \"secret trick\", \"what happened next\", \"before it\'s too late\"\n]\n\ndef check_capital_words(text):\n    words = text.split()\n    if len(words) == 0: return 0\n    return len([w for w in words if w.isupper() and len(w) > 2])\n\ndef check_exclamation_marks(text):\n    return text.count(\"!\")\n\ndef check_emotional_words(text):\n    text_lower = text.lower()\n    return [word for word in EMOTIONAL_WORDS if word in text_lower]\n\ndef check_clickbait_phrases(text):\n    text_lower = text.lower()\n    return [phrase for phrase in CLICKBAIT_PHRASES if phrase in text_lower]\n\ndef generate_linguistic_report(text):\n    reasons = []\n    capitals = check_capital_words(text)\n    exclamations = check_exclamation_marks(text)\n    emotionals = check_emotional_words(text)\n    clickbaits = check_clickbait_phrases(text)\n    \n    if capitals >= 2:\n        reasons.append(\"The text contains too many capitalized words.\")\n    if exclamations >= 2:\n        reasons.append(\"The text uses too many exclamation marks.\")\n    if len(emotionals) > 0:\n        reasons.append(\"Emotional or dramatic words detected: \" + \", \".join(emotionals))\n    if len(clickbaits) > 0:\n        reasons.append(\"Clickbait-style phrases detected: \" + \", \".join(clickbaits))\n    if len(reasons) == 0:\n        reasons.append(\"No major suspicious linguistic pattern detected.\")\n    return reasons\n\ndef calculate_linguistic_risk(text):\n    score = 0.0\n    capitals = check_capital_words(text)\n    exclamations = check_exclamation_marks(text)\n    emotionals = check_emotional_words(text)\n    clickbaits = check_clickbait_phrases(text)\n    \n    if capitals >= 2: score += 0.25\n    if exclamations >= 2: score += 0.25\n    if len(emotionals) > 0: score += 0.25\n    if len(clickbaits) > 0: score += 0.25\n    return min(score, 1.0)\n"""
with open("src/linguistic_report.py", "w") as f:
    f.write(linguistic_report_data)

predict_data = """import joblib\nimport os\nimport sys\n\nPROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), \"..\"))\nsys.path.append(os.path.join(PROJECT_ROOT, \"src\"))\n\nfrom preprocessing import clean_text\nfrom linguistic_report import generate_linguistic_report, calculate_linguistic_risk\n\ndef predict_news(text):\n    model = joblib.load(\"models/ml_model.pkl\")\n    vectorizer = joblib.load(\"models/tfidf_vectorizer.pkl\")\n    \n    cleaned = clean_text(text)\n    tfidf_feat = vectorizer.transform([cleaned])\n    prediction = model.predict(tfidf_feat)[0]\n    probs = model.predict_proba(tfidf_feat)[0]\n    \n    class_names = model.classes_\n    prob_dict = dict(zip(class_names, probs))\n    fake_prob = prob_dict.get(\"fake\", 0.0)\n    \n    ling_risk = calculate_linguistic_risk(text)\n    final_risk = (fake_prob * 0.90) + (ling_risk * 0.10)\n    report = generate_linguistic_report(text)\n    \n    return {\n        \"prediction\": prediction,\n        \"fake_probability\": fake_prob,\n        \"linguistic_risk\": ling_risk,\n        \"final_risk\": final_risk,\n        \"report\": report\n    }\n"""
with open("src/predict.py", "w") as f:
    f.write(predict_data)

print("✓ Core project files written to disk for Streamlit to access.")

### 2. Boot Streamlit & Expose public tunnel
Run the code below to boot the background server. We also output the **external IP address** which is required as a passcode by `localtunnel` when you click its generated link.

In [ ]:
# Get the passcode for localtunnel
print("=================== LOCAL TUNNEL PASSCODE ===================")
print("Copy the IP address below. You will paste it when the link opens:")
!wget -qO- ipv4.icanhazip.com
print("===========================================================\n")

# Run Streamlit in background
import subprocess
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", "8501"])
print("Streamlit process started in background.")

# Expose port using localtunnel
!npx localtunnel --port 8501